In [ ]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import StandardScaler
import warnings
warnings.filterwarnings('ignore')

LOAD THE DATA

In [ ]:
df = pd.read_csv("ppm_dummy_data.csv", low_memory=False)

print(f"Dataset loaded. Shape: {df.shape}")
print(f"Number of rows (accounts): {df.shape[0]}")
print(f"Number of columns (features): {df.shape[1]}")

REMOVAL OF LEAKAGE FEATURES

In [ ]:
# These columns are outcomes of the current billing cycle, same as the target.
# Using them would let the model "see the answer sheet" during training.
leakage_columns = [
    'Current_Payment_Amount',  # How much was paid - only knowable AFTER payment happens
    'Current_Cure',            # Whether account resolved - outcome of same cycle
]

# Identifier columns are just labels, not predictive signals.
# Account_Key is a random ID number. The model might memorise which accounts
# paid in the training data, but this would not generalise to new accounts.
identifier_columns = [
    'Account_Key',    # Unique account ID - no predictive meaning
    'ped_id_number',  # Personal ID number - identifier, not predictor
]

columns_to_drop = leakage_columns + identifier_columns
df.drop(columns=columns_to_drop, inplace=True, errors='ignore')

print(f"Dropped leakage columns: {leakage_columns}")
print(f"Dropped identifier columns: {identifier_columns}")
print(f"Dataset shape after dropping: {df.shape}")

SEPERATE 2 TARGET VARIABLES 

In [ ]:
y = df['Current_Payment'].copy()
df.drop(columns=['Current_Payment'], inplace=True)

print(f"Target variable 'Current_Payment' separated.")
print(f"\nClass distribution:")
print(y.value_counts())
print(f"\nClass balance: {y.mean()*100:.1f}% of accounts made a payment")
print(f"This is an imbalanced dataset - the model will need to handle this.")

DROP HIGH NULL COL

In [ ]:
null_rates = df.isnull().mean()
high_null_cols = null_rates[null_rates > 0.70].index.tolist()

print(f"Found {len(high_null_cols)} columns with >70% nulls:")
for col in high_null_cols:
    print(f"  {col}: {null_rates[col]*100:.1f}% missing")

In [ ]:
free_text_and_useless = [
    'Judgements1_PersonName', 'Judgements1_Plaintiff', 'Judgements1_Attorneys',
    'Deceased1_DateOfDeath', 'Deceased1_PlaceOfDeath',
    'CellNumber1_TelNumber', 'CellNumber2_TelNumber', 'CellNumber3_TelNumber',
    'WorkNumber1_TelNumber', 'WorkNumber2_TelNumber', 'WorkNumber3_TelNumber',
    'HomeNumber1_TelNumber', 'HomeNumber2_TelNumber', 'HomeNumber3_TelNumber',
    'Employer1_OriginalEmployerName', 'Employer2_OriginalEmployerName',
    'Employer1_Occupation', 'Employer2_Occupation',
    'Employer1_EmployerTelephone', 'Employer2_EmployerTelephone',
    'Person1_MarriageDate', 'EagleEye1_LastEmploymentUpdate',
    'CellNumber1_LatestDate', 'CellNumber2_LatestDate', 'CellNumber3_LatestDate',
    'WorkNumber1_LatestDate', 'WorkNumber2_LatestDate', 'WorkNumber3_LatestDate',
    'HomeNumber1_LatestDate', 'HomeNumber2_LatestDate', 'HomeNumber3_LatestDate',
    'Employer1_LatestDate', 'Employer2_LatestDate',
    'CellNumber1_TelType', 'CellNumber2_TelType', 'CellNumber3_TelType',
    'WorkNumber1_TelType', 'WorkNumber2_TelType', 'WorkNumber3_TelType',
    'HomeNumber1_TelType', 'HomeNumber2_TelType', 'HomeNumber3_TelType',
    'CreditScore1_CreditScoreCategory',
    'Address1_ComplexNumber',
    'DebtReview1_Description',
    'Book_Cycle', 'Billing_Cycle',
]

all_cols_to_drop = list(set(high_null_cols) | set(free_text_and_useless))
df.drop(columns=all_cols_to_drop, inplace=True, errors='ignore')

print(f"Dataset shape after dropping high-null and text columns: {df.shape}")

HANDLE SPEACIAL ENCODED VALUES

In [ ]:
# --- Recency ---
# A value of 9999 does NOT mean "9999 months ago".
# It means the customer has NEVER made a payment on this account.
# Strategy: Create a binary flag, then replace 9999 with a high but plausible value.

print("Recency special value (9999 = never paid):")
print(f"  Accounts with Recency = 9999: {(df['Recency'] == 9999).sum()}")

df['never_paid_flag'] = (df['Recency'] == 9999).astype(int)
df['Recency'] = df['Recency'].replace(9999, 99)

print(f"  Created 'never_paid_flag' column.")
print(f"  Replaced 9999 with 99 in Recency column.")

In [ ]:
# --- Previous_Account ---
# This flag encodes three meanings:
#   1  = account existed in the previous billing cycle
#   0  = account existed but no payment activity recorded
#  -1  = account is NEW (did not exist in the previous cycle)
#
# If we leave -1 as a number, the model treats it as "less than 0",
# which is arithmetically meaningless here. We one-hot encode it instead.

print("Previous_Account value counts:")
print(df['Previous_Account'].value_counts())

df['prev_account_new'] = (df['Previous_Account'] == -1).astype(int)
df['prev_account_existing'] = (df['Previous_Account'] == 1).astype(int)
df.drop(columns=['Previous_Account'], inplace=True)
print("\n  Created binary flags: prev_account_new, prev_account_existing")

# PAttempts and PRPCs: -1 means no prior contact history (new account)
# Replace with 0 since no contact attempts = 0 is logically equivalent
df['PAttempts'] = df['PAttempts'].replace(-1, 0)
df['PRPCs'] = df['PRPCs'].replace(-1, 0)
print("  PAttempts and PRPCs: replaced -1 with 0")

ENCODE CATEGORIAL VALUE

In [ ]:
# Gender
print("Gender unique values:", df['Gender'].unique()[:10])
df['gender_known'] = (~df['Gender'].isna()).astype(int)
df['gender_code'] = df['Gender'].fillna(-1)
df.drop(columns=['Gender'], inplace=True)
print("  Created: gender_known, gender_code")

# Drop redundant gender column from bureau data 
if 'Person1_Gender' in df.columns:
    df.drop(columns=['Person1_Gender'], inplace=True)
    print("  Dropped Person1_Gender (redundant)")

if 'Person1_MaritalStatus' in df.columns:
    df.drop(columns=['Person1_MaritalStatus'], inplace=True)
    print("  Dropped Person1_MaritalStatus (high nulls)")

# Deceased flag - True/False string to 0/1 
if 'Deceased1_IsDeceased' in df.columns:
    df['Deceased1_IsDeceased'] = df['Deceased1_IsDeceased'].map(
        {'True': 1, 'False': 0, True: 1, False: 0}).fillna(0)
    print("  Encoded Deceased1_IsDeceased as 0/1")

#  DebtReview1_StatusCode - ordinal (A < B < C < D in severity) 
if 'DebtReview1_StatusCode' in df.columns:
    status_map = {'A': 1, 'B': 2, 'C': 3, 'D': 4}
    df['debt_review_status'] = df['DebtReview1_StatusCode'].map(status_map)
    df['has_debt_review'] = df['DebtReview1_StatusCode'].notna().astype(int)
    df.drop(columns=['DebtReview1_StatusCode'], inplace=True)
    print("  Encoded DebtReview1_StatusCode into debt_review_status, has_debt_review")

# Book_key - one-hot encode (unordered category) 
if 'Book_key' in df.columns:
    book_dummies = pd.get_dummies(df['Book_key'], prefix='book', drop_first=True)
    df = pd.concat([df, book_dummies], axis=1)
    df.drop(columns=['Book_key'], inplace=True)
    print(f"  One-hot encoded Book_key into {book_dummies.shape[1]} columns")

#  Income_Band - ordinal (Low < Medium < High has a natural order) 
if 'Income_Band' in df.columns:
    income_map = {'Low': 1, 'Medium': 2, 'High': 3}
    df['income_band_encoded'] = df['Income_Band'].map(income_map)
    df.drop(columns=['Income_Band'], inplace=True)
    print("  Encoded Income_Band: Low=1, Medium=2, High=3")

# Home_Ownership_Status and Director_Status - coerce to numeric 
for col in ['Home_Ownership_Status', 'Director_Status']:
    if col in df.columns:
        df[col] = pd.to_numeric(df[col], errors='coerce')

# EagleEye boolean columns - True/False strings to 0/1
eagleeye_bool_cols = [
    'EagleEye1_HasDefault', 'EagleEye1_HasJudgment', 'EagleEye1_IsDirector',
    'EagleEye1_UnderDebtReview', 'EagleEye1_OwnsProperty',
    'EagleEye1_HasPPAccounts', 'EagleEye1_HasPPAdverse',
]
for col in eagleeye_bool_cols:
    if col in df.columns:
        df[col] = df[col].map({True: 1, False: 0, 'True': 1, 'False': 0}).fillna(0).astype(int)

if 'Income1_Confidence' in df.columns:
    df['Income1_Confidence'] = pd.to_numeric(df['Income1_Confidence'], errors='coerce')

# Drop any remaining text columns
remaining_object_cols = df.select_dtypes(include='object').columns.tolist()
if remaining_object_cols:
    print(f"\n  Dropping remaining text columns: {remaining_object_cols}")
    df.drop(columns=remaining_object_cols, inplace=True)

print(f"\nDataset shape after encoding: {df.shape}")

HANDLE MISSING VALUES

In [ ]:
remaining_nulls = df.isnull().sum()
cols_with_nulls = remaining_nulls[remaining_nulls > 0].sort_values(ascending=False)
print(f"Columns still with missing values: {len(cols_with_nulls)}")
print(cols_with_nulls)

In [ ]:
# For bureau columns with meaningful missingness, create a missing indicator flag
# BEFORE imputing. This way the model knows the value was absent.
bureau_numeric_cols = [
    'CreditScore1_CreditScore', 'Income1_Income', 'Income1_Confidence',
    'Person1_Score', 'Address1_Score',
    'CellNumber1_Score', 'CellNumber2_Score', 'CellNumber3_Score',
    'WorkNumber1_Score', 'WorkNumber2_Score', 'WorkNumber3_Score',
    'HomeNumber1_Score', 'HomeNumber2_Score', 'HomeNumber3_Score',
    'income_band_encoded', 'debt_review_status',
]

for col in bureau_numeric_cols:
    if col in df.columns and df[col].isnull().sum() > 0:
        df[col + '_missing'] = df[col].isnull().astype(int)

print("Created missing-indicator flags for bureau columns.")

# Impute all remaining numeric columns with the median
numeric_cols = df.select_dtypes(include=[np.number]).columns
for col in numeric_cols:
    if df[col].isnull().sum() > 0:
        median_val = df[col].median()
        df[col].fillna(median_val, inplace=True)

print(f"Imputed remaining nulls with column medians.")
print(f"Total missing values remaining (should be 0 or only all-null cols): {df.isnull().sum().sum()}")

SCALE NUMERIC FEATURES

In [ ]:
binary_like_cols = [col for col in df.columns
                    if set(df[col].dropna().unique()).issubset({0, 1, 0.0, 1.0})]
book_cols = [col for col in df.columns if col.startswith('book_')]
cols_to_scale = [col for col in df.select_dtypes(include=[np.number]).columns
                 if col not in binary_like_cols and col not in book_cols]

print(f"Columns to scale: {len(cols_to_scale)}")
print(f"Binary/one-hot columns left unscaled: {len(binary_like_cols) + len(book_cols)}")

scaler = StandardScaler()
df[cols_to_scale] = scaler.fit_transform(df[cols_to_scale])
print("Scaling complete.")

DROPPING NEAR CONSTANT COL

In [ ]:
near_constant = []
for col in df.columns:
    vc = df[col].value_counts(normalize=True)
    if len(vc) == 0:
        near_constant.append(col)
    elif vc.iloc[0] > 0.99:
        near_constant.append(col)

if near_constant:
    print(f"Dropping near-constant columns ({len(near_constant)}):")
    for col in near_constant:
        print(f"  {col}")
    df.drop(columns=near_constant, inplace=True)
else:
    print("No near-constant columns found.")

print(f"\nFinal feature set shape: {df.shape}")
print(f"Target variable shape: {y.shape}")

OUTLIER CAPPING

In [ ]:
# Cap extreme values using IQR
numeric_cols = df.select_dtypes(include=['int64', 'float64']).columns

for col in numeric_cols:
    Q1 = df[col].quantile(0.25)
    Q3 = df[col].quantile(0.75)
    IQR = Q3 - Q1

    lower = Q1 - 1.5 * IQR
    upper = Q3 + 1.5 * IQR

    df[col] = np.clip(df[col], lower, upper)

print("Outlier capping completed.")

MISSING PERCENTAGE REPORT

In [ ]:
missing_report = pd.DataFrame({
    'Column': df.columns,
    'Missing_%': (df.isnull().mean()*100).round(2)
})

missing_report = missing_report.sort_values(
    'Missing_%',
    ascending=False
)

print(missing_report.head(20))

CORRELATION BASED FEATURE REMOVAL

In [ ]:
corr_matrix = df.corr().abs()

upper = corr_matrix.where(
    np.triu(np.ones(corr_matrix.shape), k=1).astype(bool)
)

high_corr_features = [
    column for column in upper.columns
    if any(upper[column] > 0.95)
]

print(f"Highly correlated features: {len(high_corr_features)}")

df.drop(columns=high_corr_features, inplace=True)

In [ ]:
import os

print("Current Working Directory:")
print(os.getcwd())

print("\nFiles in Current Directory:")
for file in os.listdir():
    print(file)

In [39]:
# Save features
df.to_csv("processed_features.csv", index=False)

# Save target
y.to_frame(name="current_payment").to_csv(
    "processed_target.csv",
    index=False
)

print("Files saved successfully!")

Files saved successfully!
